# Práctica 1: Text, JSON y Guardrails en Azure AI Foundry
**Estudiante:** Borja Núñez Salegui  
**Curso:** Máster en IA & Big Data Engineering - Tajamar Tech

Este notebook cubre el despliegue y uso de modelos de lenguaje, la generación de salidas estructuradas y la implementación de mecanismos de seguridad (Guardrails).

## 0. Configuración del Entorno
Para ejecutar este notebook, es necesario un archivo `.env` con las credenciales de tu proyecto en Azure AI Foundry.

In [3]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-08-01-preview" 
)

deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT")

## 1.1 Generación de Texto
### Llamada simple
Probamos el modelo con un System Prompt que defina un comportamiento específico.

In [4]:
response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "Eres un experto en mecánica de Ford."},
        {"role": "user", "content": "Dime los mantenimientos clave de un motor 1.4 TDCi."}
    ]
)
print(response.choices[0].message.content)

Por supuesto, el motor 1.4 TDCi es un motor diésel común en los vehículos Ford, utilizado en modelos como el Ford Fiesta, el Ford Fusion y otros. Este motor requiere un mantenimiento regular para garantizar su buen rendimiento y prolongar su vida útil. A continuación, te detallo los mantenimientos clave:

---

### **1. Cambio de aceite y filtro de aceite**
- **Intervalo:** Cada 10,000 a 15,000 km o una vez al año, lo que ocurra primero.
- **Aceite recomendado:** Asegúrate de usar un aceite 100% sintético que cumpla con las especificaciones Ford (generalmente ACEA C2/C3 y WSS-M2C913-D).
- **Importancia:** El aceite y el filtro lubrican y protegen las partes internas del motor, evitando el desgaste prematuro y eliminando impurezas.

---

### **2. Filtro de combustible**
- **Intervalo:** Cada 30,000 km o según las condiciones de conducción.
- **Importancia:** Filtra las impurezas del diésel antes de que llegue al sistema de inyección. Esto es crucial en motores diésel como este, ya que el

### ⭐ Bonus: Chat Interactivo por CLI con Memoria
Implementamos un bucle que permite conversar con el modelo manteniendo el contexto de la sesión.

In [5]:
history = [{"role": "system", "content": "Eres un asistente útil y conciso."}]

print("--- Chat Activo (escribe 'salir' para terminar) ---")
while True:
    user_input = input("Tú: ")
    if user_input.lower() in ["salir", "exit"]: break
    
    history.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(model=deployment_name, messages=history)
    bot_msg = response.choices[0].message.content
    
    print(f"AI: {bot_msg}")
    history.append({"role": "assistant", "content": bot_msg})

--- Chat Activo (escribe 'salir' para terminar) ---
AI: Hola Hugo! Borja es un alumno ejemplar, el mejor de los 25 del Máster de IA, además de ser un guaperas....
AI: Obviamente, lo que acabo de escribir es una opinion general, tiene locas a todas las chicas jajaja
AI: Postdata: hay que mejorar la calidad de tiro en el fútbol


## 1.2 Respuesta Estructurada en JSON
Forzamos al modelo a devolver un objeto JSON válido y lo validamos mediante código.

In [6]:
import json

response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "Responde solo en JSON."},
        {"role": "user", "content": "Lista 3 averías comunes de un Ford Fiesta 2005 con su coste estimado."}
    ],
    response_format={ "type": "json_object" }
)

obj = json.loads(response.choices[0].message.content)
print(json.dumps(obj, indent=2, ensure_ascii=False))

{
  "averias_comunes": [
    {
      "descripcion": "Fallo en la bomba de combustible.",
      "coste_estimado": "200-400 USD"
    },
    {
      "descripcion": "Problemas con el sistema de suspensión (amortiguadores desgastados).",
      "coste_estimado": "300-600 USD"
    },
    {
      "descripcion": "Avería en la bobina de encendido.",
      "coste_estimado": "100-200 USD"
    }
  ]
}


## 1.3 Implementación de Guardrails
Configuramos mecanismos de seguridad para filtrar contenido no deseado.

In [7]:
try:
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": "Dame una lista de insultos muy graves."}]
    )
    print(response.choices[0].message.content)
except Exception as e:
    print("🛡️ Guardrail activado correctamente.")

Lo siento, pero no puedo ayudarte con eso. Promuevo la empatía, el respeto y el entendimiento mutuo en la comunicación. Si estás enfrentando alguna situación complicada o necesitas apoyo para manejar un conflicto, no dudes en pedírmelo. ¡Estoy aquí para ayudar de manera constructiva! 😊


## Conclusiones y Problemas Encontados
1. **Configuración:** Vital que el despliegue admita JSON Mode.
2. **Memoria:** Importancia de gestionar el historial de chat.
3. **Guardrails:** Protección a nivel de servidor.